In [ ]:
print('Installing ffmpeg...')
!apt-get update
!apt-get install -y ffmpeg
print('ffmpeg installed.')

Installing ffmpeg...
Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:9 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,703 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,297 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,972 kB]
Get:13 http://security.ubuntu

In [ ]:
print('Installing yt-dlp and openai-whisper...')
!pip install yt-dlp openai-whisper
print('yt-dlp and openai-whisper installed.')

Installing yt-dlp and openai-whisper...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 35.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 114.3 MB/s eta 0:00:00
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=a7073e5e394ea206bea8e1a403c8870d1b7fe0fe6ff2c793270c00e461e8870d
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper
yt-dlp and openai-whisper installed.


In [ ]:
print('Installing deno (JavaScript runtime) as suggested by yt-dlp...')
!curl -fsSL https://deno.land/x/install/install.sh | sh
import os
os.environ['DENO_INSTALL'] = '/root/.deno'
os.environ['PATH'] = f'{os.environ["DENO_INSTALL"]}/bin:{os.environ["PATH"]}'
print('deno installed.')

Installing deno (JavaScript runtime) as suggested by yt-dlp...
######################################################################## 100.0%
Archive:  /root/.deno/bin/deno.zip
  inflating: /root/.deno/bin/deno    
Installed dx alias, if this conflicts with an existing command, you can remove it with `rm $(which dx)` and choose a new name with `dx --install-alias <new-name>`
Deno was installed successfully to /root/.deno/bin/deno
sh: 109: cannot open /dev/tty: No such device or address
deno installed.


In [ ]:
import yt_dlp
import whisper
import os
import re

# ==========================================================
# 1. ADD YOUR YOUTUBE LINKS HERE
# ==========================================================
# You can add as many links as you want, separated by commas.
video_links = [
    "https://www.youtube.com/watch?v=ET8rwerJTg0",
    # "https://www.youtube.com/watch?v=EXAMPLE_2",  <-- Add more links like this
    # "https://www.youtube.com/watch?v=EXAMPLE_3",
]

# ==========================================================
# 2. INITIALIZE THE AI MODEL
# ==========================================================
print("Loading Whisper 'medium' model onto GPU...")
# "medium" is excellent for Hindi/multilingual accuracy.
# Swap to "small" if you want it to run faster but with slightly less precision.
model = whisper.load_model("medium")
print("Model loaded successfully.\n" + "="*50)

# Helper function to clean video titles so they can safely be used as filenames
def clean_filename(title):
    return re.sub(r'[\\/*?:"<>|]', "", title).replace(" ", "_")

# ==========================================================
# 3. LOOP THROUGH EACH LINK
# ==========================================================
for i, url in enumerate(video_links, start=1):
    print(f"\nProcessing Video {i} of {len(video_links)}: {url}")

    # Configure downloader to get the title first, then extract audio
    ydl_opts = {
        'format': 'bestaudio/best',
        'outtmpl': 'temp_audio_track.%(ext)s',  # Generic temporary name
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'mp3',
            'preferredquality': '192',
        }],
        'quiet': True, # Keeps the console clean
        'geo_bypass_country': 'US', # Attempt to bypass geo-restrictions
        'user_agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36', # Common User-Agent
        'allow_anonymous_ownership_info': True, # Try to bypass some authentication checks
    }

    try:
        # Fetch video metadata (like the actual title)
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info_dict = ydl.extract_info(url, download=False)
            video_title = info_dict.get('title', f'video_{i}')
            safe_title = clean_filename(video_title)

            print(f"-> Title found: '{video_title}'")
            print("-> Downloading audio stream...")
            ydl.download([url])

        temp_audio = "temp_audio_track.mp3"

        if os.path.exists(temp_audio):
            print("-> Audio downloaded. Running AI Speech-to-Text...")

            # Run transcription (language="hi" forces Hindi output)
            # Remove language="hi" if your future links are strictly English videos
            result = model.transcribe(temp_audio, language="hi")

            # Create a dedicated filename for this specific video
            output_filename = f"{safe_title}_transcript.txt"

            # Save the text file
            with open(output_filename, "w", encoding="utf-8") as f:
                f.write(result["text"])

            print(f"✓ Success! Saved transcript to: '{output_filename}'")

            # Clean up the temporary audio file to save Colab disk space
            os.remove(temp_audio)
        else:
            print(f"✗ Error: Failed to generate audio for video {i}")

    except Exception as e:
        print(f"✗ An error occurred while processing this link: {e}")
        # Continue to the next video even if one fails
        if os.path.exists("temp_audio_track.mp3"):
            os.remove("temp_audio_track.mp3")
        continue

print("\n" + "="*50 + "\nAll batch transcriptions are complete!")

Loading Whisper 'medium' model onto GPU...
Model loaded successfully.

Processing Video 1 of 1: https://www.youtube.com/watch?v=ET8rwerJTg0


ERROR: [youtube] ET8rwerJTg0: Sign in to confirm you’re not a bot. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies


✗ An error occurred while processing this link: ERROR: [youtube] ET8rwerJTg0: Sign in to confirm you’re not a bot. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies

All batch transcriptions are complete!
